In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Netflix Recommendation System") \
    .getOrCreate()

print("Spark Ready")

Spark Ready


In [8]:
movies = spark.read.csv(
    r"D:\GitHub\DPySpark\data\movies.csv",
    header=True,
    inferSchema=True
)

ratings = spark.read.csv(
    r"D:\GitHub\DPySpark\data\ratings.csv",
    header=True,
    inferSchema=True
)

In [9]:
ratings.groupBy("userId") \
       .count() \
       .orderBy("count", ascending=False) \
       .show(20)

+------+-----+
|userId|count|
+------+-----+
| 72315|32202|
| 80974| 9178|
|137293| 8913|
| 33844| 7919|
| 20055| 7488|
|109731| 6647|
| 92046| 6564|
| 49403| 6553|
| 30879| 5693|
|115102| 5649|
|110971| 5633|
| 75309| 5525|
| 78849| 5276|
| 61010| 5244|
| 29803| 5219|
|122011| 5160|
| 57548| 5066|
| 93855| 5045|
|103611| 4861|
| 34987| 4831|
+------+-----+
only showing top 20 rows


In [10]:
ratings.groupBy("movieId") \
       .count() \
       .orderBy("count", ascending=False) \
       .show(20)

+-------+-----+
|movieId|count|
+-------+-----+
|    356|81491|
|    318|81482|
|    296|79672|
|    593|74127|
|   2571|72674|
|    260|68717|
|    480|64144|
|    527|60411|
|    110|59184|
|   2959|58773|
|    589|57379|
|   1196|57361|
|      1|57309|
|   4993|55736|
|     50|55366|
|   1210|54917|
|   1198|54675|
|   2858|53689|
|    858|52498|
|   5952|51138|
+-------+-----+
only showing top 20 rows


In [11]:
movie_ratings = ratings.join(
    movies,
    on="movieId",
    how="inner"
)

movie_ratings.show(5)

+-------+------+------+----------+--------------------+--------------------+
|movieId|userId|rating| timestamp|               title|              genres|
+-------+------+------+----------+--------------------+--------------------+
|    296|     1|   5.0|1147880044| Pulp Fiction (1994)|Comedy|Crime|Dram...|
|    306|     1|   3.5|1147868817|Three Colors: Red...|               Drama|
|    307|     1|   5.0|1147868828|Three Colors: Blu...|               Drama|
|    665|     1|   5.0|1147878820|  Underground (1995)|    Comedy|Drama|War|
|    899|     1|   3.5|1147868510|Singin' in the Ra...|Comedy|Musical|Ro...|
+-------+------+------+----------+--------------------+--------------------+
only showing top 5 rows


In [12]:
movie_ratings.groupBy("title") \
             .count() \
             .orderBy("count", ascending=False) \
             .show(20, truncate=False)

+------------------------------------------------------------------------------+-----+
|title                                                                         |count|
+------------------------------------------------------------------------------+-----+
|Forrest Gump (1994)                                                           |81491|
|Shawshank Redemption, The (1994)                                              |81482|
|Pulp Fiction (1994)                                                           |79672|
|Silence of the Lambs, The (1991)                                              |74127|
|Matrix, The (1999)                                                            |72674|
|Star Wars: Episode IV - A New Hope (1977)                                     |68717|
|Jurassic Park (1993)                                                          |64144|
|Schindler's List (1993)                                                       |60411|
|Braveheart (1995)                         

In [13]:
movie_ratings.groupBy("title") \
             .avg("rating") \
             .orderBy("avg(rating)", ascending=False) \
             .show(20, truncate=False)

+-----------------------------------------+-----------+
|title                                    |avg(rating)|
+-----------------------------------------+-----------+
|Hitlar (1980)                            |5.0        |
|A Police Inspector Calls (1974)          |5.0        |
|Adrenaline (1990)                        |5.0        |
|Renaldo and Clara (1978)                 |5.0        |
|The Boxer (1972)                         |5.0        |
|We Are Mountains (1969)                  |5.0        |
|Underground (2011)                       |5.0        |
|Tarzan's Fight for Life (1958)           |5.0        |
|La muerte de Jaime Roldós (2013)         |5.0        |
|Turbulence 3: Heavy Metal (2001)         |5.0        |
|Aschenputtel (2010)                      |5.0        |
|The Monkey King 3 (2018)                 |5.0        |
|Injecting Aluminum                       |5.0        |
|Neapolitan Diary (1992)                  |5.0        |
|We All Fall Down (2016)                  |5.0  

In [14]:
from pyspark.sql.functions import avg, count

movie_ratings.groupBy("title") \
    .agg(
        avg("rating").alias("avg_rating"),
        count("rating").alias("num_ratings")
    ) \
    .filter("num_ratings > 1000") \
    .orderBy("avg_rating", ascending=False) \
    .show(20, truncate=False)

+---------------------------------------------------------------------------+------------------+-----------+
|title                                                                      |avg_rating        |num_ratings|
+---------------------------------------------------------------------------+------------------+-----------+
|Planet Earth II (2016)                                                     |4.483096085409253 |1124       |
|Planet Earth (2006)                                                        |4.464796794504865 |1747       |
|Shawshank Redemption, The (1994)                                           |4.413576004516335 |81482      |
|Band of Brothers (2001)                                                    |4.398598820058997 |1356       |
|Godfather, The (1972)                                                      |4.324336165187245 |52498      |
|Usual Suspects, The (1995)                                                 |4.284353213163313 |55366      |
|Godfather: Part II